# NB12 — Reviewer 2: strict removal of cross-experiment exact spectral overlap

**Project:** `BANANA_RIPENESS_MULTISENSOR_Q1`

This notebook performs the final duplicate/leakage robustness control requested after NB11.

## Objective

NB11 identified exactly two 18-band spectral signatures that occur in more than one experiment:
- one signature appears in Tests 3 and 4 and is associated with stages 4 and 5;
- one signature appears in Tests 3 and 4 and is associated only with stage 5.

This notebook **does not hard-code their row locations**. It recomputes exact spectral groups from the frozen master dataset, identifies all groups that occur in more than one experiment, verifies the NB11 finding, removes **all rows belonging to those groups from every experiment**, and then repeats the nested LOEO analysis.

## Representations

After removing all cross-experiment exact spectral overlaps and collapsing exact within-experiment vector–stage repeats, four spectral representations are evaluated:

1. `RAW_NOSHARED_DEDUP`
2. `SNV_NOSHARED_DEDUP`
3. `DOMAIN_Z_ROW_NOSHARED_DEDUP`
4. `DOMAIN_Z_UNIQUE_NOSHARED_DEDUP`

## Scientific guardrails

- The frozen RAW/NB10/NB11 results are read only for comparison.
- All shared spectral signatures are excluded **before** normalization, model fitting, tuning, or evaluation.
- Label conflicts are never resolved by majority vote.
- DOMAIN_Z uses unlabeled target-domain feature statistics only and remains a transductive diagnostic.
- Hyperparameters and model family are re-selected using the original nested LOEO rule after removal.

**Hardware:** CPU only.


In [ ]:
# 0. Mount Drive, dependencies and configuration
from google.colab import drive
drive.mount('/content/drive')

import sys, subprocess, importlib.util
if importlib.util.find_spec('mord') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mord==0.7'])
if importlib.util.find_spec('xgboost') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'xgboost'])

from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, os, platform, shutil, time, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score,
    matthews_corrcoef, cohen_kappa_score, mean_absolute_error
)

import mord
import xgboost as xgb

warnings.filterwarnings('ignore')

SEED=2026
np.random.seed(SEED)

PROJECT=Path('/content/drive/MyDrive/Paper3_Banano/BANANA_RIPENESS_MULTISENSOR_Q1')
MASTER_PATH=PROJECT/'02_EXTRACTED_DATA'/'06_master_original_multisensor.csv'
PROTOCOL_PATH=PROJECT/'04_FROZEN_SPLITS'/'nested_loeo_protocol_v1.json'
RESULTS_ROOT=PROJECT/'06_RESULTS'

assert MASTER_PATH.exists(),MASTER_PATH
assert PROTOCOL_PATH.exists(),PROTOCOL_PATH
assert RESULTS_ROOT.exists(),RESULTS_ROOT

RUN_ID=datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_DIR=RESULTS_ROOT/f'NB12_REVIEWER2_REMOVE_SHARED_SPECTRAL_{RUN_ID}'
TABLE_DIR=RUN_DIR/'tables'
FIG_DIR=RUN_DIR/'figures'
TEXT_DIR=RUN_DIR/'text'
META_DIR=RUN_DIR/'metadata'
for d in [TABLE_DIR,FIG_DIR,TEXT_DIR,META_DIR]:
    d.mkdir(parents=True,exist_ok=True)

ALL_LABELS=[1,2,3,4,5]
OUTER_TESTS=[2,3,4,5]
MODELS=['OrdinalLogistic','PLSDA','SVM_RBF','RandomForest','XGBoost']
METHODS=[
    'RAW_NOSHARED_DEDUP',
    'SNV_NOSHARED_DEDUP',
    'DOMAIN_Z_ROW_NOSHARED_DEDUP',
    'DOMAIN_Z_UNIQUE_NOSHARED_DEDUP'
]
N_JOBS=max(1,min(8,(os.cpu_count() or 2)-1))

start_all=time.perf_counter()

print('Project:',PROJECT)
print('Output:',RUN_DIR)
print('CPU workers:',N_JOBS)
print('Methods:',METHODS)


In [ ]:
# 1. Frozen provenance and previous-run references
def sha256_file(path,chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(chunk_size),b''):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_MASTER_SHA='0a06fb3773f85dacee42e18eecf17e5e2a8e7f5e9508cd3d60bd4b66757e8ea8'
EXPECTED_PROTOCOL_SHA='86731ff9a292f1afd35a215105b14c4af3aa2bda5e8c4dad4df639d4d82ad930'

assert sha256_file(MASTER_PATH)==EXPECTED_MASTER_SHA
assert sha256_file(PROTOCOL_PATH)==EXPECTED_PROTOCOL_SHA

LATEST_NB10=RESULTS_ROOT/'LATEST_NB10_REVIEWER2_RUN.json'
LATEST_NB11=RESULTS_ROOT/'LATEST_NB11_REVIEWER2_RUN.json'
assert LATEST_NB10.exists(),LATEST_NB10
assert LATEST_NB11.exists(),LATEST_NB11

nb10_ptr=json.loads(LATEST_NB10.read_text(encoding='utf-8'))
nb11_ptr=json.loads(LATEST_NB11.read_text(encoding='utf-8'))

NB10_RUN=Path(nb10_ptr['run_dir'])
NB11_RUN=Path(nb11_ptr['run_dir'])

NB10_SELECTED=NB10_RUN/'tables'/'09_selected_model_performance_raw_snv_domainz.csv'
NB11_SELECTED=NB11_RUN/'tables'/'12_selected_model_duplicate_aware.csv'
NB11_OVERLAP=NB11_RUN/'tables'/'02_cross_experiment_exact_spectral_overlap.csv'

for p in [NB10_SELECTED,NB11_SELECTED,NB11_OVERLAP]:
    assert p.exists(),p
    print('Reference:',p)

source_hashes=pd.DataFrame([
    {'artifact':str(MASTER_PATH.relative_to(PROJECT)),
     'sha256':sha256_file(MASTER_PATH),'bytes':MASTER_PATH.stat().st_size},
    {'artifact':str(PROTOCOL_PATH.relative_to(PROJECT)),
     'sha256':sha256_file(PROTOCOL_PATH),'bytes':PROTOCOL_PATH.stat().st_size},
    {'artifact':str(NB10_SELECTED.relative_to(PROJECT)),
     'sha256':sha256_file(NB10_SELECTED),'bytes':NB10_SELECTED.stat().st_size},
    {'artifact':str(NB11_SELECTED.relative_to(PROJECT)),
     'sha256':sha256_file(NB11_SELECTED),'bytes':NB11_SELECTED.stat().st_size},
    {'artifact':str(NB11_OVERLAP.relative_to(PROJECT)),
     'sha256':sha256_file(NB11_OVERLAP),'bytes':NB11_OVERLAP.stat().st_size},
])
source_hashes.to_csv(META_DIR/'source_hashes.csv',index=False)

print('Frozen provenance: PASS')


In [ ]:
# 2. Recompute exact spectral groups and verify NB11 shared signatures
df=pd.read_csv(MASTER_PATH,low_memory=False).reset_index(drop=True)
df['test_id']=df['test_id'].astype(int)
df['stage']=df['stage'].astype(int)

SPECTRAL_FEATURES=[
    'SpA410','SpB435','SpC460','SpD485','SpE510','SpF535',
    'SpG560','SpH585','SpR610','SpI645','SpS680','SpJ705',
    'SpT730','SpU760','SpV810','SpW860','SpK900','SpL940'
]

assert len(df)==17662
assert df['master_id'].is_unique
assert df[SPECTRAL_FEATURES].notna().all().all()

df['spectral_group_global']=df.groupby(
    SPECTRAL_FEATURES,sort=False,dropna=False
).ngroup()

overlap=(df.groupby('spectral_group_global',as_index=False)
         .agg(
             n_rows=('master_id','size'),
             n_tests=('test_id','nunique'),
             n_stages=('stage','nunique'),
             tests=('test_id',lambda x:','.join(map(str,sorted(set(x))))),
             stages=('stage',lambda x:','.join(map(str,sorted(set(x))))),
         ))

shared=overlap[overlap.n_tests>1].copy().sort_values('spectral_group_global').reset_index(drop=True)

# Strict reproducibility check against NB11.
nb11_shared=pd.read_csv(NB11_OVERLAP).sort_values('spectral_group_global').reset_index(drop=True)

assert len(shared)==2,shared
assert shared[['spectral_group_global','n_rows','n_tests','n_stages','tests','stages']].equals(
    nb11_shared[['spectral_group_global','n_rows','n_tests','n_stages','tests','stages']]
)

assert shared.n_rows.tolist()==[156,122]
assert shared.tests.tolist()==['3,4','3,4']

shared.to_csv(TABLE_DIR/'01_recomputed_cross_experiment_overlap.csv',index=False)

print('Shared spectral groups verified:')
display(shared)
print('Rows to remove:',int(shared.n_rows.sum()))


In [ ]:
# 3. Remove every row belonging to any cross-experiment shared spectral group
shared_ids=set(shared.spectral_group_global.astype(int).tolist())
remove_mask=df.spectral_group_global.isin(shared_ids)

removed=df.loc[remove_mask,[
    'master_id','test_id','stage','spectral_group_global'
]+SPECTRAL_FEATURES].copy()

clean=df.loc[~remove_mask].copy().reset_index(drop=True)

assert len(removed)==278
assert len(clean)==17384

# Recompute group IDs after removal only for bookkeeping.
clean['spectral_group_clean']=clean.groupby(
    SPECTRAL_FEATURES,sort=False,dropna=False
).ngroup()

remaining_overlap=(clean.groupby('spectral_group_clean')
                   .test_id.nunique())
assert int((remaining_overlap>1).sum())==0

removed.to_csv(TABLE_DIR/'02_removed_shared_spectral_rows.csv.gz',index=False,compression='gzip')

removal_counts=(removed.groupby(['spectral_group_global','test_id','stage'],as_index=False)
                .size()
                .rename(columns={'size':'n_rows_removed'}))
removal_counts.to_csv(TABLE_DIR/'03_removed_rows_by_group_test_stage.csv',index=False)

print('Rows before:',len(df))
print('Rows removed:',len(removed))
print('Rows retained:',len(clean))
print('Remaining cross-experiment exact spectral groups: 0')
display(removal_counts)


In [ ]:
# 4. Collapse exact within-experiment vector-stage repeats after shared-group removal
dedup=clean.drop_duplicates(
    subset=['test_id']+SPECTRAL_FEATURES+['stage'],
    keep='first'
).copy().reset_index(drop=True)

assert len(dedup)==1895

sample_sizes=pd.DataFrame({
    'test_id':OUTER_TESTS,
    'original_rows':[int((df.test_id==t).sum()) for t in OUTER_TESTS],
    'rows_after_shared_removal':[int((clean.test_id==t).sum()) for t in OUTER_TESTS],
    'dedup_vector_stage_rows':[int((dedup.test_id==t).sum()) for t in OUTER_TESTS],
})
sample_sizes['retained_after_shared_removal']=(
    sample_sizes.rows_after_shared_removal/sample_sizes.original_rows
)
sample_sizes['retained_after_dedup']=(
    sample_sizes.dedup_vector_stage_rows/sample_sizes.original_rows
)
sample_sizes.to_csv(TABLE_DIR/'04_sample_size_after_shared_removal.csv',index=False)

print('Duplicate-collapsed rows after shared-group removal:',len(dedup))
display(sample_sizes.round(4))


In [ ]:
# 5. Preprocessing representations
def snv(X):
    X=np.asarray(X,float)
    mu=X.mean(axis=1,keepdims=True)
    sd=X.std(axis=1,ddof=1,keepdims=True)
    sd=np.where((~np.isfinite(sd)) | (sd<1e-12),1.0,sd)
    return (X-mu)/sd

Xraw=dedup[SPECTRAL_FEATURES].to_numpy(float)
TRANSFORMED={}

TRANSFORMED['RAW_NOSHARED_DEDUP']=Xraw.copy()
TRANSFORMED['SNV_NOSHARED_DEDUP']=snv(Xraw)

X_row=np.empty_like(Xraw,dtype=float)
X_unique=np.empty_like(Xraw,dtype=float)

domain_stats=[]

for t in OUTER_TESTS:
    idx=np.where(dedup.test_id.to_numpy(int)==t)[0]

    # ROW variant: all retained unlabeled rows from this experiment.
    full_t=clean[clean.test_id.eq(t)]
    fullX=full_t[SPECTRAL_FEATURES].to_numpy(float)
    mu_row=fullX.mean(axis=0,keepdims=True)
    sd_row=fullX.std(axis=0,ddof=0,keepdims=True)
    sd_row=np.where((~np.isfinite(sd_row)) | (sd_row<1e-12),1.0,sd_row)
    X_row[idx]=(Xraw[idx]-mu_row)/sd_row

    # UNIQUE variant: each retained unlabeled spectral signature contributes once.
    unique_t=full_t.drop_duplicates(subset=SPECTRAL_FEATURES)
    uniqueX=unique_t[SPECTRAL_FEATURES].to_numpy(float)
    mu_unique=uniqueX.mean(axis=0,keepdims=True)
    sd_unique=uniqueX.std(axis=0,ddof=0,keepdims=True)
    sd_unique=np.where((~np.isfinite(sd_unique)) | (sd_unique<1e-12),1.0,sd_unique)
    X_unique[idx]=(Xraw[idx]-mu_unique)/sd_unique

    domain_stats.append({
        'test_id':t,
        'n_rows_row_stats':len(fullX),
        'n_unique_vectors_unique_stats':len(uniqueX),
        'mean_abs_difference_band_means':float(np.mean(np.abs(mu_row-mu_unique))),
        'mean_abs_difference_band_sds':float(np.mean(np.abs(sd_row-sd_unique))),
    })

TRANSFORMED['DOMAIN_Z_ROW_NOSHARED_DEDUP']=X_row
TRANSFORMED['DOMAIN_Z_UNIQUE_NOSHARED_DEDUP']=X_unique

for method,X in TRANSFORMED.items():
    assert X.shape==(len(dedup),18)
    assert np.isfinite(X).all()

domain_stats=pd.DataFrame(domain_stats)
domain_stats.to_csv(TABLE_DIR/'05_domain_z_stats_after_shared_removal.csv',index=False)
display(domain_stats.round(4))


In [ ]:
# 6. Metrics, models and original compact NB03 grid
def classification_metrics(y_true,y_pred):
    y_true=np.asarray(y_true,int)
    y_pred=np.asarray(y_pred,int)
    observed=sorted(np.unique(y_true).tolist())
    ae=np.abs(y_true-y_pred)
    return {
        'accuracy':float(accuracy_score(y_true,y_pred)),
        'macro_f1_observed':float(f1_score(y_true,y_pred,labels=observed,average='macro',zero_division=0)),
        'macro_f1_fixed_1to5':float(f1_score(y_true,y_pred,labels=ALL_LABELS,average='macro',zero_division=0)),
        'balanced_accuracy':float(balanced_accuracy_score(y_true,y_pred)),
        'mcc':float(matthews_corrcoef(y_true,y_pred)),
        'qwk':float(cohen_kappa_score(y_true,y_pred,labels=ALL_LABELS,weights='quadratic')),
        'ordinal_mae':float(mean_absolute_error(y_true,y_pred)),
        'within_1_stage':float(np.mean(ae<=1)),
        'severe_error_ge2':float(np.mean(ae>=2)),
        'observed_labels':','.join(map(str,observed)),
    }

def fit_predict(model_name,params,X_train,y_train,X_eval,seed):
    y_train=np.asarray(y_train,int)

    if model_name=='OrdinalLogistic':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train)
        Xq=scaler.transform(X_eval)
        model=mord.LogisticAT(alpha=float(params['alpha']))
        model.fit(Xt,y_train)
        pred=model.predict(Xq).astype(int)

    elif model_name=='PLSDA':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train)
        Xq=scaler.transform(X_eval)
        classes=np.sort(np.unique(y_train))
        Y=np.zeros((len(y_train),len(classes)),float)
        for j,c in enumerate(classes):
            Y[:,j]=(y_train==c).astype(float)
        ncomp=int(min(params['n_components'],Xt.shape[1],Xt.shape[0]-1))
        model=PLSRegression(n_components=ncomp,scale=False,max_iter=1000)
        model.fit(Xt,Y)
        scores=np.asarray(model.predict(Xq))
        if scores.ndim==1:
            scores=scores[:,None]
        pred=classes[np.argmax(scores,axis=1)].astype(int)

    elif model_name=='SVM_RBF':
        scaler=StandardScaler().fit(X_train)
        Xt=scaler.transform(X_train)
        Xq=scaler.transform(X_eval)
        model=SVC(
            kernel='rbf',
            C=float(params['C']),
            gamma=params['gamma'],
            probability=False,
            cache_size=1000,
            random_state=seed
        )
        model.fit(Xt,y_train)
        pred=model.predict(Xq).astype(int)

    elif model_name=='RandomForest':
        model=RandomForestClassifier(
            n_estimators=int(params['n_estimators']),
            max_depth=params['max_depth'],
            min_samples_leaf=int(params['min_samples_leaf']),
            max_features=params['max_features'],
            random_state=seed,
            n_jobs=N_JOBS
        )
        model.fit(X_train,y_train)
        pred=model.predict(X_eval).astype(int)

    elif model_name=='XGBoost':
        classes=np.sort(np.unique(y_train))
        cmap={c:i for i,c in enumerate(classes)}
        yz=np.array([cmap[c] for c in y_train],int)
        model=xgb.XGBClassifier(
            objective='multi:softprob',
            num_class=len(classes),
            n_estimators=int(params['n_estimators']),
            max_depth=int(params['max_depth']),
            learning_rate=float(params['learning_rate']),
            subsample=float(params['subsample']),
            colsample_bytree=float(params['colsample_bytree']),
            reg_lambda=1.0,
            min_child_weight=1.0,
            tree_method='hist',
            eval_metric='mlogloss',
            random_state=seed,
            n_jobs=N_JOBS
        )
        model.fit(X_train,yz,verbose=False)
        pred=classes[np.asarray(model.predict(X_eval),int)].astype(int)

    else:
        raise ValueError(model_name)

    return np.clip(pred,1,5).astype(int)

def candidate_grid(model_name):
    if model_name=='OrdinalLogistic':
        return [{'alpha':0.01},{'alpha':0.1},{'alpha':1.0},{'alpha':10.0}]
    if model_name=='PLSDA':
        return [{'n_components':2},{'n_components':4},{'n_components':8}]
    if model_name=='SVM_RBF':
        return [
            {'C':0.5,'gamma':'scale'},
            {'C':2.0,'gamma':'scale'},
            {'C':8.0,'gamma':'scale'},
            {'C':8.0,'gamma':0.03},
        ]
    if model_name=='RandomForest':
        return [
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':1,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':18,'min_samples_leaf':1,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':3,'max_features':'sqrt'},
            {'n_estimators':400,'max_depth':None,'min_samples_leaf':1,'max_features':0.7},
        ]
    if model_name=='XGBoost':
        return [
            {'n_estimators':250,'max_depth':3,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':350,'max_depth':5,'learning_rate':0.05,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':250,'max_depth':3,'learning_rate':0.10,'subsample':0.9,'colsample_bytree':0.9},
            {'n_estimators':500,'max_depth':2,'learning_rate':0.03,'subsample':0.9,'colsample_bytree':0.9},
        ]
    raise ValueError(model_name)

assert sum(len(candidate_grid(m)) for m in MODELS)==19
print('Model implementations and grid ready.')


## Full nested LOEO after strict overlap removal

Expected inner fits:

4 preprocessing arms × 4 outer folds × 3 inner validation experiments × 19 candidates = **912 inner fits**.

The modeling dataset now contains **1,895 vector–stage observations** and no exact spectral signature occurs in more than one experiment.


In [ ]:
# 7. Inner LOEO
inner_rows=[]
leakage_rows=[]
fit_counter=0
inner_start=time.perf_counter()

test_arr=dedup.test_id.to_numpy(int)
y_all=dedup.stage.to_numpy(int)

for method_idx,method in enumerate(METHODS):
    Xall=TRANSFORMED[method]

    for outer_test in OUTER_TESTS:
        dev_tests=[t for t in OUTER_TESTS if t!=outer_test]
        outer_mask=(test_arr==outer_test)

        print(f'\n{method} | outer Test {outer_test} | dev={dev_tests}')

        for model_idx,model_name in enumerate(MODELS):
            for cand_idx,params in enumerate(candidate_grid(model_name)):
                pjson=json.dumps(params,sort_keys=True)

                for inner_val_test in dev_tests:
                    inner_train_tests=[t for t in dev_tests if t!=inner_val_test]
                    tr=np.isin(test_arr,inner_train_tests)
                    va=(test_arr==inner_val_test)

                    assert not np.any(tr & va)
                    assert not np.any(tr & outer_mask)
                    assert not np.any(va & outer_mask)

                    seed=SEED+method_idx*10000+outer_test*100+inner_val_test*10+cand_idx

                    t0=time.perf_counter()
                    status='ok'; err=''
                    try:
                        predv=fit_predict(
                            model_name,params,
                            Xall[tr],y_all[tr],
                            Xall[va],seed
                        )
                        met=classification_metrics(y_all[va],predv)
                    except Exception as e:
                        status='error'; err=repr(e)
                        met={k:np.nan for k in [
                            'accuracy','macro_f1_observed','macro_f1_fixed_1to5',
                            'balanced_accuracy','mcc','qwk','ordinal_mae',
                            'within_1_stage','severe_error_ge2'
                        ]}
                        met['observed_labels']=','.join(
                            map(str,sorted(np.unique(y_all[va]).tolist()))
                        )

                    sec=time.perf_counter()-t0
                    fit_counter+=1

                    inner_rows.append({
                        'preprocessing':method,
                        'outer_test':outer_test,
                        'inner_val_test':inner_val_test,
                        'inner_train_tests':','.join(map(str,inner_train_tests)),
                        'model':model_name,
                        'candidate_id':cand_idx,
                        'params_json':pjson,
                        'n_train':int(tr.sum()),
                        'n_val':int(va.sum()),
                        'status':status,
                        'error':err,
                        'fit_seconds':sec,
                        **met
                    })

                    leakage_rows.append({
                        'preprocessing':method,
                        'outer_test':outer_test,
                        'inner_val_test':inner_val_test,
                        'model':model_name,
                        'candidate_id':cand_idx,
                        'outer_rows_used_in_train':int(np.sum(tr & outer_mask)),
                        'outer_rows_used_in_validation':int(np.sum(va & outer_mask)),
                        'train_val_overlap_rows':int(np.sum(tr & va)),
                    })

                    if fit_counter%100==0:
                        print(f'  {fit_counter}/912 inner fits | elapsed {(time.perf_counter()-inner_start)/60:.1f} min')

inner=pd.DataFrame(inner_rows)
leak=pd.DataFrame(leakage_rows)

inner.to_csv(
    TABLE_DIR/'06_inner_cv_fold_results_no_shared.csv.gz',
    index=False,
    compression='gzip'
)
leak.to_csv(TABLE_DIR/'07_inner_cv_leakage_audit.csv',index=False)

assert fit_counter==912
assert inner.status.eq('ok').all(),inner.loc[~inner.status.eq('ok'),['preprocessing','outer_test','model','error']]
assert leak[['outer_rows_used_in_train','outer_rows_used_in_validation','train_val_overlap_rows']].to_numpy().sum()==0

print('\nInner LOEO complete.')
print('Leakage audit: PASS')
print('Elapsed minutes:',round((time.perf_counter()-inner_start)/60,2))


In [ ]:
# 8. Candidate aggregation and original NB03 lexicographic selection
agg=(inner.groupby(
    ['preprocessing','outer_test','model','candidate_id','params_json'],
    as_index=False
).agg(
    inner_folds=('inner_val_test','nunique'),
    mean_macro_f1_observed=('macro_f1_observed','mean'),
    sd_macro_f1_observed=('macro_f1_observed','std'),
    worst_macro_f1_observed=('macro_f1_observed','min'),
    mean_macro_f1_fixed_1to5=('macro_f1_fixed_1to5','mean'),
    mean_balanced_accuracy=('balanced_accuracy','mean'),
    mean_mcc=('mcc','mean'),
    mean_qwk=('qwk','mean'),
    mean_ordinal_mae=('ordinal_mae','mean'),
    mean_within_1_stage=('within_1_stage','mean'),
    mean_severe_error_ge2=('severe_error_ge2','mean'),
    total_fit_seconds=('fit_seconds','sum')
))
agg['eligible']=agg.inner_folds.eq(3)
agg.to_csv(TABLE_DIR/'08_inner_candidate_summary_no_shared.csv',index=False)

selected_rows=[]
for keys,g in agg[agg.eligible].groupby(['preprocessing','outer_test','model'],sort=True):
    gs=g.sort_values(
        ['mean_macro_f1_observed','worst_macro_f1_observed','mean_qwk','mean_ordinal_mae','candidate_id'],
        ascending=[False,False,False,True,True],
        kind='mergesort'
    )
    selected_rows.append(gs.iloc[0].copy())

selected_hp=pd.DataFrame(selected_rows).reset_index(drop=True)
assert len(selected_hp)==4*4*5

selected_hp['inner_model_rank']=selected_hp.groupby(
    ['preprocessing','outer_test']
).mean_macro_f1_observed.rank(method='min',ascending=False)

selected_hp.to_csv(TABLE_DIR/'09_selected_hyperparameters_no_shared.csv',index=False)
display(selected_hp.head(10))


In [ ]:
# 9. Outer LOEO
outer_rows=[]
outer_fit_counter=0
outer_start=time.perf_counter()

for method_idx,method in enumerate(METHODS):
    Xall=TRANSFORMED[method]

    for outer_test in OUTER_TESTS:
        tr=(test_arr!=outer_test)
        te=(test_arr==outer_test)

        for model_idx,model_name in enumerate(MODELS):
            srow=selected_hp[
                selected_hp.preprocessing.eq(method) &
                selected_hp.outer_test.eq(outer_test) &
                selected_hp.model.eq(model_name)
            ]
            assert len(srow)==1
            s=srow.iloc[0]
            params=json.loads(s.params_json)

            seed=SEED+method_idx*10000+outer_test*100+model_idx*10

            t0=time.perf_counter()
            status='ok'; err=''
            try:
                predv=fit_predict(
                    model_name,params,
                    Xall[tr],y_all[tr],
                    Xall[te],seed
                )
                met=classification_metrics(y_all[te],predv)
            except Exception as e:
                status='error'; err=repr(e)
                met={k:np.nan for k in [
                    'accuracy','macro_f1_observed','macro_f1_fixed_1to5',
                    'balanced_accuracy','mcc','qwk','ordinal_mae',
                    'within_1_stage','severe_error_ge2'
                ]}
                met['observed_labels']=','.join(
                    map(str,sorted(np.unique(y_all[te]).tolist()))
                )

            outer_fit_counter+=1

            outer_rows.append({
                'preprocessing':method,
                'outer_test':outer_test,
                'model':model_name,
                'candidate_id':int(s.candidate_id),
                'params_json':s.params_json,
                'inner_mean_macro_f1':float(s.mean_macro_f1_observed),
                'inner_worst_macro_f1':float(s.worst_macro_f1_observed),
                'inner_model_rank':float(s.inner_model_rank),
                'n_train':int(tr.sum()),
                'n_test':int(te.sum()),
                'status':status,
                'error':err,
                'fit_seconds':time.perf_counter()-t0,
                **met
            })

            print(
                f'{method:32s} | T{outer_test} | {model_name:15s} | '
                f'Macro-F1={met["macro_f1_observed"]:.4f}'
            )

outer=pd.DataFrame(outer_rows)
outer.to_csv(TABLE_DIR/'10_outer_loeo_metrics_no_shared.csv',index=False)

assert outer_fit_counter==80
assert outer.status.eq('ok').all(),outer.loc[~outer.status.eq('ok'),['preprocessing','outer_test','model','error']]

print('\nOuter LOEO complete.')
print('Elapsed minutes:',round((time.perf_counter()-outer_start)/60,2))


In [ ]:
# 10. Internally selected model after shared-group removal
selected_rows=[]

for keys,g in outer.groupby(['preprocessing','outer_test'],sort=True):
    gs=g.sort_values(
        ['inner_mean_macro_f1','inner_worst_macro_f1','model'],
        ascending=[False,False,True],
        kind='mergesort'
    )
    selected_rows.append(gs.iloc[0].copy())

selected=pd.DataFrame(selected_rows)[[
    'preprocessing','outer_test','model',
    'inner_mean_macro_f1','macro_f1_observed',
    'macro_f1_fixed_1to5','balanced_accuracy','mcc','qwk','ordinal_mae','n_test'
]].reset_index(drop=True)

assert len(selected)==16
selected.to_csv(TABLE_DIR/'11_selected_model_no_shared.csv',index=False)
display(selected.round(4))


In [ ]:
# 11. Compare directly with corresponding NB11 duplicate-aware results
nb11=pd.read_csv(NB11_SELECTED)

method_map={
    'RAW_NOSHARED_DEDUP':'RAW_DEDUP',
    'SNV_NOSHARED_DEDUP':'SNV_DEDUP',
    'DOMAIN_Z_ROW_NOSHARED_DEDUP':'DOMAIN_Z_ROW_DEDUP',
    'DOMAIN_Z_UNIQUE_NOSHARED_DEDUP':'DOMAIN_Z_UNIQUE_DEDUP'
}

compare_rows=[]

for nb12_method,nb11_method in method_map.items():
    a=selected[selected.preprocessing.eq(nb12_method)].copy()
    b=nb11[nb11.preprocessing.eq(nb11_method)].copy()

    z=a.merge(
        b[['outer_test','model','macro_f1_observed','balanced_accuracy','mcc','qwk','ordinal_mae']],
        on='outer_test',
        suffixes=('_nb12','_nb11'),
        validate='one_to_one'
    )

    for r in z.itertuples():
        compare_rows.append({
            'outer_test':int(r.outer_test),
            'nb12_method':nb12_method,
            'nb11_method':nb11_method,
            'nb12_selected_model':r.model_nb12,
            'nb11_selected_model':r.model_nb11,
            'macro_f1_nb11':float(r.macro_f1_observed_nb11),
            'macro_f1_nb12':float(r.macro_f1_observed_nb12),
            'delta_nb12_minus_nb11':float(r.macro_f1_observed_nb12-r.macro_f1_observed_nb11),
            'balanced_accuracy_nb11':float(r.balanced_accuracy_nb11),
            'balanced_accuracy_nb12':float(r.balanced_accuracy_nb12),
            'mcc_nb11':float(r.mcc_nb11),
            'mcc_nb12':float(r.mcc_nb12),
            'qwk_nb11':float(r.qwk_nb11),
            'qwk_nb12':float(r.qwk_nb12),
            'ordinal_mae_nb11':float(r.ordinal_mae_nb11),
            'ordinal_mae_nb12':float(r.ordinal_mae_nb12),
        })

compare=pd.DataFrame(compare_rows)
compare.to_csv(TABLE_DIR/'12_nb12_vs_nb11_selected_comparison.csv',index=False)

summary=(compare.groupby(['nb12_method','nb11_method'],as_index=False)
         .agg(
             mean_macro_f1_nb11=('macro_f1_nb11','mean'),
             mean_macro_f1_nb12=('macro_f1_nb12','mean'),
             mean_delta=('delta_nb12_minus_nb11','mean'),
             median_delta=('delta_nb12_minus_nb11','median'),
             max_abs_delta=('delta_nb12_minus_nb11',lambda x:float(np.max(np.abs(x)))),
             min_delta=('delta_nb12_minus_nb11','min'),
             max_delta=('delta_nb12_minus_nb11','max'),
             selected_model_matches=('nb12_selected_model',
                 lambda x:0)
         ))

# Recompute model-match counts directly.
for i,row in summary.iterrows():
    g=compare[
        compare.nb12_method.eq(row.nb12_method) &
        compare.nb11_method.eq(row.nb11_method)
    ]
    summary.loc[i,'selected_model_matches']=int(
        (g.nb12_selected_model==g.nb11_selected_model).sum()
    )

summary.to_csv(TABLE_DIR/'13_nb12_vs_nb11_summary.csv',index=False)

display(compare.round(4))
print('\nSummary:')
display(summary.round(4))


In [ ]:
# 12. Final domain-normalization comparison and figures
dz=selected[
    selected.preprocessing.isin([
        'DOMAIN_Z_ROW_NOSHARED_DEDUP',
        'DOMAIN_Z_UNIQUE_NOSHARED_DEDUP'
    ])
].copy()

# Figure A — NB11 vs NB12 for all four methods
fig,axes=plt.subplots(2,2,figsize=(12,9),sharex=True,sharey=True)
axes=axes.ravel()

for ax,(nb12_method,nb11_method) in zip(axes,method_map.items()):
    a=selected[selected.preprocessing.eq(nb12_method)].sort_values('outer_test')
    b=nb11[nb11.preprocessing.eq(nb11_method)].sort_values('outer_test')

    ax.plot(b.outer_test,b.macro_f1_observed,marker='o',label='NB11: shared signatures retained')
    ax.plot(a.outer_test,a.macro_f1_observed,marker='s',label='NB12: shared signatures removed')
    ax.set_title(nb12_method.replace('_NOSHARED_DEDUP',''))
    ax.set_xticks(OUTER_TESTS)
    ax.grid(alpha=0.2)

for ax in axes[2:]:
    ax.set_xlabel('Held-out experiment')
for ax in axes[::2]:
    ax.set_ylabel('Selected-model external Macro-F1')

handles,labels=axes[0].get_legend_handles_labels()
fig.legend(handles,labels,loc='lower center',ncol=2,bbox_to_anchor=(0.5,-0.01))
fig.suptitle('Sensitivity to complete removal of cross-experiment exact spectral signatures',y=1.01)
fig.tight_layout(rect=[0,0.05,1,1])

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2G_remove_cross_experiment_overlap.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)

# Figure B — deltas
plot=compare.pivot_table(
    index='nb12_method',
    columns='outer_test',
    values='delta_nb12_minus_nb11'
).reindex(METHODS)

fig,ax=plt.subplots(figsize=(9,4.8))
im=ax.imshow(plot.to_numpy(),aspect='auto')
ax.set_xticks(range(len(plot.columns)),[f'Test {c}' for c in plot.columns])
ax.set_yticks(
    range(len(plot.index)),
    [m.replace('_NOSHARED_DEDUP','') for m in plot.index]
)
ax.set_title('Change after removing all cross-experiment exact spectral signatures')

for i in range(plot.shape[0]):
    for j in range(plot.shape[1]):
        ax.text(j,i,f'{plot.iloc[i,j]:+.3f}',ha='center',va='center')

fig.colorbar(im,ax=ax,label='Δ Macro-F1: NB12 − NB11')
fig.tight_layout()

for ext in ['png','pdf','svg']:
    fig.savefig(
        FIG_DIR/f'Figure_R2H_delta_after_overlap_removal.{ext}',
        dpi=300 if ext=='png' else None,
        bbox_inches='tight'
    )
plt.close(fig)


In [ ]:
# 13. Key results and manuscript-ready notes
key={
    'n_cross_experiment_shared_groups_removed':int(len(shared)),
    'n_rows_removed':int(len(removed)),
    'n_rows_remaining_before_within_experiment_dedup':int(len(clean)),
    'n_vector_stage_rows_after_dedup':int(len(dedup)),
    'remaining_cross_experiment_exact_spectral_groups':0,
    'mean_selected_macro_f1_nb12':{
        method:float(
            selected[selected.preprocessing.eq(method)].macro_f1_observed.mean()
        )
        for method in METHODS
    },
    'comparison_with_nb11':{
        row.nb12_method:{
            'mean_macro_f1_nb11':float(row.mean_macro_f1_nb11),
            'mean_macro_f1_nb12':float(row.mean_macro_f1_nb12),
            'mean_delta':float(row.mean_delta),
            'max_abs_delta':float(row.max_abs_delta),
            'selected_model_matches_of_4':int(row.selected_model_matches),
        }
        for row in summary.itertuples()
    }
}

(TABLE_DIR/'14_key_results_reviewer2_remove_shared.json').write_text(
    json.dumps(key,indent=2,ensure_ascii=False),
    encoding='utf-8'
)

notes=[
    '# Reviewer 2 — strict exclusion of cross-experiment exact spectral signatures',
    '',
    '## Removal audit',
    f'- Exact spectral signatures occurring in more than one experiment: {len(shared)}.',
    f'- Original rows belonging to these signatures and removed before preprocessing: {len(removed)}.',
    f'- Rows remaining before within-experiment duplicate collapse: {len(clean)}.',
    f'- Vector–stage rows remaining after duplicate collapse: {len(dedup)}.',
    '- Remaining exact spectral signatures shared across experiments: 0.',
    '',
    '## Mean selected external Macro-F1 after complete overlap removal'
]

for method in METHODS:
    g=selected[selected.preprocessing.eq(method)]
    notes.append(
        f'- {method}: {g.macro_f1_observed.mean():.4f}.'
    )

notes += [
    '',
    '## Change relative to NB11'
]

for r in summary.itertuples():
    notes.append(
        f'- {r.nb12_method}: mean Δ Macro-F1={r.mean_delta:+.4f}; '
        f'max |Δ| across held-out experiments={r.max_abs_delta:.4f}; '
        f'same selected model in {int(r.selected_model_matches)}/4 outer folds.'
    )

notes += [
    '',
    '## Interpretation rule',
    'If NB12 remains close to NB11, the duplicate-aware spectral-normalization conclusion is robust '
    'to complete removal of all exact spectral signatures shared between experiments. This directly '
    'rules out exact cross-experiment feature-vector overlap as the primary explanation for the '
    'DOMAIN_Z improvement.',
    '',
    'If performance changes materially, the manuscript must report the overlap-sensitive estimate '
    'instead of claiming robustness.',
    '',
    'DOMAIN_Z remains transductive because it uses unlabeled target-batch feature statistics. '
    'No target labels are used.'
]

(TEXT_DIR/'reviewer2_remove_shared_overlap_notes.md').write_text(
    '\n'.join(notes)+'\n',
    encoding='utf-8'
)

print('\n'.join(notes))


In [ ]:
# 14. Manifest, hashes, ZIP and automatic download
versions={
    'python':platform.python_version(),
    'numpy':np.__version__,
    'pandas':pd.__version__,
    'matplotlib':plt.matplotlib.__version__,
    'scikit_learn':sklearn.__version__,
    'mord':getattr(mord,'__version__','0.7/unknown'),
    'xgboost':xgb.__version__,
}

elapsed=time.perf_counter()-start_all

manifest={
    'project':'BANANA_RIPENESS_MULTISENSOR_Q1',
    'notebook':'12_REVIEWER2_REMOVE_CROSS_EXPERIMENT_EXACT_SPECTRAL_OVERLAP',
    'run_id':RUN_ID,
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'seed':SEED,
    'source_hashes':source_hashes.to_dict('records'),
    'nb10_reference_run':str(NB10_RUN),
    'nb11_reference_run':str(NB11_RUN),
    'shared_group_detection':'exact equality across all 18 spectral bands; group appears in >1 test_id',
    'n_shared_groups_removed':int(len(shared)),
    'n_original_rows_removed':int(len(removed)),
    'remaining_shared_groups':0,
    'dedup_rule':'one row per experiment + exact 18-band vector + observed stage; conflicts retained',
    'methods':METHODS,
    'models':MODELS,
    'selection_rule':[
        'maximize mean observed-label Macro-F1 across 3 inner experiment folds',
        'maximize worst inner-fold observed-label Macro-F1',
        'maximize mean QWK',
        'minimize mean ordinal MAE'
    ],
    'inner_fits':int(fit_counter),
    'outer_fits':int(outer_fit_counter),
    'outer_label_use_for_preprocessing':False,
    'domain_z_target_feature_use':'unlabeled target-domain features only',
    'elapsed_minutes':elapsed/60,
    'versions':versions,
    'key_results':key
}

(META_DIR/'run_manifest.json').write_text(
    json.dumps(manifest,indent=2,ensure_ascii=False),
    encoding='utf-8'
)

(META_DIR/'environment.txt').write_text(
    '\n'.join([
        f'platform={platform.platform()}',
        f'cpu_count={os.cpu_count()}',
        *[f'{k}={v}' for k,v in versions.items()]
    ])+'\n',
    encoding='utf-8'
)

readme=[
    '# NB12 Reviewer 2 — remove cross-experiment exact spectral overlap',
    '',
    f'Run ID: `{RUN_ID}`',
    f'Runtime: {elapsed/60:.2f} min',
    '',
    f'- Shared exact spectral signatures removed: {len(shared)}',
    f'- Original rows removed: {len(removed)}',
    '- Remaining cross-experiment exact spectral overlap: 0',
    '- Full nested LOEO selection repeated after removal',
    '- RAW, SNV, DOMAIN_Z-row and DOMAIN_Z-unique compared with NB11',
    '- No target labels used for preprocessing or selection',
    '',
    'Upload the generated ZIP to ChatGPT for final interpretation.'
]

(RUN_DIR/'README_RESULTS.md').write_text(
    '\n'.join(readme)+'\n',
    encoding='utf-8'
)

hash_rows=[]
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file() and p.name!='package_hashes.csv':
        hash_rows.append({
            'relative_path':str(p.relative_to(RUN_DIR)),
            'bytes':p.stat().st_size,
            'sha256':sha256_file(p)
        })

pd.DataFrame(hash_rows).to_csv(
    META_DIR/'package_hashes.csv',
    index=False
)

zip_base=RESULTS_ROOT/f'RESULTS_NB12_REVIEWER2_REMOVE_SHARED_SPECTRAL_{RUN_ID}'
zip_path=Path(shutil.make_archive(
    str(zip_base),'zip',
    root_dir=RUN_DIR.parent,
    base_dir=RUN_DIR.name
))

latest={
    'notebook':'12_REVIEWER2_REMOVE_CROSS_EXPERIMENT_EXACT_SPECTRAL_OVERLAP',
    'run_id':RUN_ID,
    'run_dir':str(RUN_DIR),
    'zip_path':str(zip_path),
    'zip_bytes':zip_path.stat().st_size,
    'elapsed_minutes':elapsed/60
}

(RESULTS_ROOT/'LATEST_NB12_REVIEWER2_RUN.json').write_text(
    json.dumps(latest,indent=2),
    encoding='utf-8'
)

print('\n✅ NB12 COMPLETED')
print('Run directory:',RUN_DIR)
print('ZIP:',zip_path)
print('ZIP size (MB):',round(zip_path.stat().st_size/1024**2,2))
print('Elapsed minutes:',round(elapsed/60,2))
print('\nUpload this ZIP to ChatGPT.')

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Automatic browser download did not complete:',repr(e))
    print('ZIP remains safely stored in Drive:',zip_path)
